# DVCM Physical Verification

Independent checks (not JSON trace replay): mass-conservation growth steps and post-collapse head rise vs discrete collision estimate. Mirrors **`tests/test_dvcm_physical_verification.py`**.

> **DVCM regression traces:** [`dvcm_canonical_verification.ipynb`](dvcm_canonical_verification.ipynb)

## 1. Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import rthym_moc
from _verification_notebook_setup import bootstrap_verification_notebook

REPO_ROOT, TESTS_DIR = bootstrap_verification_notebook()
print(f"Repository root: {REPO_ROOT}")
print(f"rthym_moc: {getattr(rthym_moc, '__version__', 'unknown')}")

## 2. Run canonical rapid-recovery transient

In [ ]:
from dvcm_physical_verification_utils import (
    COLLAPSE_SPIKE_RTOL,
    DEFAULT_DT_S,
    MASS_STEP_ATOL_FT3,
    evaluate_collapse_spike,
    evaluate_mass_conservation,
    run_physical_verification_case,
)

results = run_physical_verification_case(dt=DEFAULT_DT_S)
mass = evaluate_mass_conservation(results, dt=DEFAULT_DT_S, atol_ft3=MASS_STEP_ATOL_FT3)
spike = evaluate_collapse_spike(results, dt=DEFAULT_DT_S, rtol=COLLAPSE_SPIKE_RTOL)
print(f"Mass conservation: PASS={mass.passed} ({mass.n_steps_checked} steps, max err {mass.max_abs_step_error_ft3:.3e} ft^3)")
print(f"Collapse spike: PASS={spike.passed} rel_err={spike.relative_error:.3f}")

## 3. Cavity volume and junction head

In [ ]:
t = np.asarray(results["time"], dtype=float)
vol = np.asarray(results["node_cavity_volume"]["J1"], dtype=float)
head = np.asarray(results["node_head"]["J1"], dtype=float)
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
axes[0].plot(t, vol, color="darkcyan")
axes[0].set_ylabel("Cavity volume (ft³)")
axes[0].grid(True, alpha=0.3)
axes[1].plot(t, head, color="firebrick")
axes[1].axvline(spike.collapse_step * DEFAULT_DT_S, color="orange", linestyle=":", label="Primary collapse")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("J1 head (ft)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 4. Summary

In [ ]:
overall = mass.passed and spike.passed
print("Overall:", "PASS" if overall else "FAIL")